# Time-series analysis — QuPath measurements (multi-plate)

**Template** — edit the *Configuration* cell and run all cells.

For experiments spread across multiple Incucyte vessels (plates), each mapped
by a `plate_map.csv` that includes a `plate` column. Plate indices are mapped
to vessel IDs (`VIDxxxx`) via the `PLATE_IDS` list.

Input: single QuPath TSV containing measurements from all plates (VID parsed
from each filename) + `plate_map.csv`.
Output: per-plate QC, per-condition time-series with 95 % CI across biological replicates.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path().resolve().parents[1]  # repo root
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import hcs_analysis as hcs


## Configuration


In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
EXP_DIR       = ROOT / "data" / "TODO_experiment"
RESULTS_TSV   = EXP_DIR / "Results" / "measurements.tsv"  # all plates concatenated
PLATE_MAP_CSV = EXP_DIR / "plate_map.csv"

# Vessel IDs in plate order (plate 1 first) — must match the `plate` column in plate_map.csv
PLATE_IDS = ["VID0000", "VID0001"]  # replace with actual VID numbers

# Time index parameters — required when filenames have no embedded timestamp
# Set INTERVAL_MIN = None if timestamps are in the Incucyte filenames.
INTERVAL_MIN = None   # minutes between consecutive QuPath time indices
START_MIN    = 0      # elapsed_min for time index 0

CLASSIFICATION = None        # keep only one class, e.g. "Neurosphere"; None keeps all
METRICS = ["Area px^2", "Circularity"]  # measurement columns to plot

CONDITIONS_ORDER = ["Control", "TODO"]  # left → right in plots
# ────────────────────────────────────────────────────────────────────────────

## 1. Load and combine plates


In [ ]:
df = hcs.load_qupath(RESULTS_TSV, interval_min=INTERVAL_MIN, start_min=START_MIN)

plate_map = hcs.load_plate_map(PLATE_MAP_CSV, plate_ids=PLATE_IDS)
df = hcs.merge_plate_map(df, plate_map)

if CLASSIFICATION is not None:
    df = hcs.filter_by(df, Classification=CLASSIFICATION)

print(f"{len(df):,} objects  |  {df['Well'].nunique()} wells  |  "
      f"{df['PlateID'].nunique()} plates  |  "
      f"{df['elapsed_min'].nunique()} timepoints")
df.head(3)

## 2. QC — object counts


In [ ]:
# Per-well counts (summed across all timepoints)
for plate_id, grp in df.groupby("PlateID"):
    fig, ax = plt.subplots(figsize=(12, 4))
    hcs.qc_counts(grp, groupby="Well", ax=ax)
    ax.set_title(f"Object count per well — {plate_id} (all timepoints)")
    plt.tight_layout()
    plt.show()


## 3. Aggregate

Two steps:
1. `to_well_means` — per-object → per-well mean at each timepoint
2. Average technical replicates → one value per (sample, condition, timepoint)


In [ ]:
df_well = hcs.to_well_means(df, metrics=METRICS)

df_bio = (
    df_well
    .groupby(["sample", "condition", "elapsed_min"])[METRICS]
    .mean()
    .reset_index()
)
df_bio["elapsed_h"] = df_bio["elapsed_min"] / 60

print("Biological replicates per condition:")
print(df_bio.groupby("condition")["sample"].nunique())
df_bio.head(3)


## 4. Time-series plots

One line per condition; shaded band = 95 % bootstrapped CI across biological replicates.


In [ ]:
# Filter to desired conditions order if needed
df_plot = df_bio[df_bio["condition"].isin(CONDITIONS_ORDER)].copy()

fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5))
if len(METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, METRICS):
    hcs.time_series(df_plot, y=metric, time_col="elapsed_h", ax=ax)
    ax.set_xlabel("Time (h)")
    ax.set_title(metric)
plt.tight_layout()
